# ARCHIVE — superseded diagnostics and `W_noise` training sweeps

**Do not reuse these results as evidence.** Kept only as a record of what was run
and why it was abandoned. Nothing here is required by the reproduction notebook
or by Phase A.

These cells were written against a live runtime and reference names and files that
may no longer exist (`patch_records`, old sweep checkpoints, `comparison_results.json`).
Several will not run at all. That is expected.

## Why these were set aside

**Laplacian / high-frequency proxy diagnostics.** A Laplacian or Sobel proxy responds
to genuine texture, edges, sharpening and clipping just as strongly as to noise, so it
is not a calibrated noise measurement. Superseded by the controlled synthetic-noise
measurements in Phase A. One result is still worth remembering:

> `Pearson r(error, input noise, flat) = 0.065` — noise amplification does **not**
> meaningfully drive full-reference error.

**`W_noise` sweeps.** A training-side `L_noise` penalised `relu(gain - 1)` on a
Sobel-magnitude flatness mask.

| run | PSNR | SSIM | MAE |
|---|---|---|---|
| baseline | 17.26 | 0.556 | 32.91 |
| W50 (buggy mask) | 13.93 | 0.541 | 52.72 |
| W50 (fixed mask) | 14.74 | 0.555 | 47.91 |
| W400 | 9.21 | 0.325 | 86.57 |

The v1 sweep (0.1–2.0) did nothing: at `W_noise=2.0` the term was 0.67% of total loss.
The v2/v3 sweeps (20–400) moved the proxy and BRISQUE but dimmed the image and
degraded every full-reference metric.

**This was not a mask bug.** The local gain `g` *is* the contrast gain, so penalising
`g` penalises enhancement. The model satisfied the penalty by darkening the output —
exactly as asked. Retuning the weight or the mask cannot fix an objective that is
pointed at the wrong quantity. Two consequences for future work:

1. Do not reuse `W50` or any weight from these sweeps.
2. Do not build a regulariser that penalises `g^2` or `J^2`; it will fail the same way.

A secondary issue: the mask used Sobel gradient **magnitude**, which is large for
edges *and* for noise and cannot separate them. Structure-tensor **coherence** can.

## Part 1 — Laplacian / high-frequency proxy diagnostics (cells 14–19)

In [ ]:
import os, sys, glob, json
import numpy as np
import torch
from PIL import Image
from scipy.ndimage import sobel, laplace

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

DCE_net = model.enhance_net_nopool().cuda().eval()
DCE_net.load_state_dict(torch.load("/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth", map_location="cuda"))

PATCH = 16              # matches the paper's own local-region size for L_exp
GRAD_PERCENTILE = 30    # bottom 30% gradient-magnitude patches per image = "flat"

def to_gray(img01):
    return img01.mean(axis=2)

def crop_to_grid(arr, patch=PATCH):
    H, W = arr.shape[:2]
    return arr[: (H // patch) * patch, : (W // patch) * patch]

def patch_reduce(arr2d, patch, fn):
    H, W = arr2d.shape
    nph, npw = H // patch, W // patch
    reshaped = arr2d[: nph * patch, : npw * patch].reshape(nph, patch, npw, patch)
    return fn(reshaped, axis=(1, 3))

def run_model(image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, _, r = DCE_net(t)
    r_iters = r.squeeze(0).cpu().numpy().reshape(8, 3, *r.shape[-2:])
    x = t.squeeze(0).cpu().numpy()  # 3xHxW in [0,1]
    stages = [x]
    for i in range(8):
        x = x + r_iters[i] * (np.power(x, 2) - x)
        stages.append(x.copy())
    stages_hwc = [np.transpose(s, (1, 2, 0)) for s in stages]  # 9 x HxWx3 (stage 0 = input)
    avg_alpha = np.abs(r_iters).mean(axis=(0, 1))  # HxW, averaged over 8 iters and RGB
    return stages_hwc, avg_alpha

test_images = sorted(glob.glob("data/test_data/DICM/*.jpg")) + sorted(glob.glob("data/test_data/LIME/*.bmp"))
print(f"{len(test_images)} test images")

patch_records = []
for path in test_images:
    stages_hwc, avg_alpha = run_model(path)
    x0 = crop_to_grid(stages_hwc[0])
    avg_alpha = crop_to_grid(avg_alpha)
    gray_in = to_gray(x0)
    grad = np.hypot(sobel(gray_in, axis=1), sobel(gray_in, axis=0))
    grad_patch = patch_reduce(grad, PATCH, np.mean)
    thresh = np.percentile(grad_patch, GRAD_PERCENTILE)
    flat_mask = grad_patch <= thresh

    darkness_patch = 1 - patch_reduce(gray_in, PATCH, np.mean)
    alpha_patch = patch_reduce(avg_alpha, PATCH, np.mean)

    noise_by_stage = []
    for stage_img in stages_hwc:
        gray_stage = to_gray(crop_to_grid(stage_img))
        lap = laplace(gray_stage)
        noise_by_stage.append(patch_reduce(lap, PATCH, np.std))

    nph, npw = flat_mask.shape
    for i in range(nph):
        for j in range(npw):
            if not flat_mask[i, j]:
                continue
            patch_records.append({
                "file": os.path.basename(path),
                "alpha": float(alpha_patch[i, j]),
                "darkness": float(darkness_patch[i, j]),
                "noise": [float(noise_by_stage[s][i, j]) for s in range(9)],
            })

print(f"{len(patch_records)} flat patches collected across {len(test_images)} images")

DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/diagnostics"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
with open(f"{DRIVE_RESULTS}/patch_records.json", "w") as f:
    json.dump(patch_records, f)
print("Saved raw data to", DRIVE_RESULTS)

In [ ]:
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

alpha = np.array([p["alpha"] for p in patch_records])
darkness = np.array([p["darkness"] for p in patch_records])
noise_in = np.array([p["noise"][0] for p in patch_records])
noise_out = np.array([p["noise"][8] for p in patch_records])

amp_ratio = noise_out / (noise_in + 1e-6)
log_amp = np.log(amp_ratio + 1e-3)

r_alpha, p_alpha = pearsonr(alpha, log_amp)
r_dark, p_dark = pearsonr(darkness, log_amp)
print(f"Pearson r(log amplification ratio, mean curve |A|) = {r_alpha:.3f}  (p={p_alpha:.2e})")
print(f"Pearson r(log amplification ratio, input darkness)  = {r_dark:.3f}  (p={p_dark:.2e})")
print(f"median amplification ratio: {np.median(amp_ratio):.2f}x   (n={len(amp_ratio)} flat patches)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(alpha, log_amp, s=4, alpha=0.25)
axes[0].set_xlabel("mean curve parameter |A| (flat patch, avg over 8 iters, RGB)")
axes[0].set_ylabel("log noise amplification ratio")
axes[0].set_title(f"Test 1a: r={r_alpha:.3f}")

axes[1].scatter(darkness, log_amp, s=4, alpha=0.25)
axes[1].set_xlabel("input darkness (1 - mean luma)")
axes[1].set_ylabel("log noise amplification ratio")
axes[1].set_title(f"Test 1b: r={r_dark:.3f}")
plt.tight_layout()

DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/diagnostics"
plt.savefig(f"{DRIVE_RESULTS}/test1_noise_vs_alpha.png", dpi=120)
plt.show()
print("Saved to", DRIVE_RESULTS)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

noise_matrix = np.array([p["noise"] for p in patch_records])  # n_patches x 9
mean_per_stage = noise_matrix.mean(axis=0)
sem_per_stage = noise_matrix.std(axis=0) / np.sqrt(len(patch_records))

print("Mean flat-patch noise level by stage (0=input, 1-8=after curve iteration i):")
for i, v in enumerate(mean_per_stage):
    print(f"  stage {i}: {v:.5f}")
print("\nStage-over-stage growth:", np.round(np.diff(mean_per_stage), 5))

plt.figure(figsize=(7, 5))
plt.errorbar(range(9), mean_per_stage, yerr=sem_per_stage, marker="o", capsize=3)
plt.xlabel("curve iteration (0 = input)")
plt.ylabel("mean flat-patch noise level (± SEM)")
plt.title(f"Noise growth across Zero-DCE's 8 curve iterations (n={len(patch_records)} flat patches)")
plt.grid(alpha=0.3)
plt.savefig(f"{DRIVE_RESULTS}/test2_noise_per_iteration.png", dpi=120)
plt.show()
print("Saved to", DRIVE_RESULTS)

In [ ]:
import os, sys, json
import numpy as np
from PIL import Image
from scipy.ndimage import sobel, laplace
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")

PATCH = 16
GRAD_PERCENTILE = 30
DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/diagnostics"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
LOL_RESULTS = "/content/drive/MyDrive/zero-dce/results/lol_eval15"

LOL_ROOT = "/content/lol_dataset/lol_dataset/eval15"
if not os.path.isdir(LOL_ROOT):
    from huggingface_hub import hf_hub_download
    import shutil
    zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset", filename="lol_dataset.zip", repo_type="dataset")
    shutil.unpack_archive(zip_path, "/content/lol_dataset")

LOW_DIR, HIGH_DIR = f"{LOL_ROOT}/low", f"{LOL_ROOT}/high"
low_files = sorted(f for f in os.listdir(LOW_DIR) if not f.startswith("."))

def to_gray(a): return a.mean(axis=2)
def crop_to_grid(a, patch=PATCH):
    H, W = a.shape[:2]
    return a[: (H // patch) * patch, : (W // patch) * patch]
def patch_reduce(a, patch, fn):
    H, W = a.shape
    nph, npw = H // patch, W // patch
    r = a[: nph * patch, : npw * patch].reshape(nph, patch, npw, patch)
    return fn(r, axis=(1, 3))

all_patches = []
for fname in low_files:
    x0 = np.asarray(Image.open(f"{LOW_DIR}/{fname}").convert("RGB")) / 255.0
    H, W = x0.shape[:2]
    gt_img = Image.open(f"{HIGH_DIR}/{fname}").convert("RGB")
    enh_img = Image.open(f"{LOL_RESULTS}/{fname}").convert("RGB")
    if gt_img.size != (W, H): gt_img = gt_img.resize((W, H))
    if enh_img.size != (W, H): enh_img = enh_img.resize((W, H))
    gt = np.asarray(gt_img).astype(np.float64)
    enh = np.asarray(enh_img).astype(np.float64)

    x0, gt, enh = crop_to_grid(x0), crop_to_grid(gt), crop_to_grid(enh)
    gray_in = to_gray(x0)
    grad = np.hypot(sobel(gray_in, axis=1), sobel(gray_in, axis=0))
    grad_patch = patch_reduce(grad, PATCH, np.mean)
    flat_mask = grad_patch <= np.percentile(grad_patch, GRAD_PERCENTILE)

    darkness_patch = 1 - patch_reduce(gray_in, PATCH, np.mean)
    noise_in_patch = patch_reduce(laplace(gray_in), PATCH, np.std)
    err_patch = patch_reduce(np.abs(enh - gt).mean(axis=2), PATCH, np.mean)

    nph, npw = flat_mask.shape
    for i in range(nph):
        for j in range(npw):
            all_patches.append({"file": fname, "darkness": float(darkness_patch[i, j]),
                                 "flat": bool(flat_mask[i, j]), "noise_in": float(noise_in_patch[i, j]),
                                 "err": float(err_patch[i, j])})

print(f"{len(all_patches)} patches across {len(low_files)} LOL eval15 images")

darkness = np.array([p["darkness"] for p in all_patches])
err = np.array([p["err"] for p in all_patches])
flat = np.array([p["flat"] for p in all_patches])
noise_in = np.array([p["noise_in"] for p in all_patches])

r_dark, p_dark = pearsonr(darkness, err)
print(f"Pearson r(error, darkness), all patches = {r_dark:.3f}  (p={p_dark:.2e})")
r_noise, p_noise = pearsonr(noise_in[flat], err[flat])
print(f"Pearson r(error, input noise), flat patches only = {r_noise:.3f}  (p={p_noise:.2e})  (n={flat.sum()})")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(darkness, err, s=6, alpha=0.3)
axes[0].set_xlabel("input darkness (1 - mean luma)"); axes[0].set_ylabel("|enhanced - GT| (0-255)")
axes[0].set_title(f"Test 3a: r={r_dark:.3f}")
axes[1].scatter(noise_in[flat], err[flat], s=6, alpha=0.3)
axes[1].set_xlabel("input noise level (flat patches)"); axes[1].set_ylabel("|enhanced - GT| (0-255)")
axes[1].set_title(f"Test 3b: r={r_noise:.3f}")
plt.tight_layout()
plt.savefig(f"{DRIVE_RESULTS}/test3_error_vs_darkness_noise.png", dpi=120)
plt.show()

with open(f"{DRIVE_RESULTS}/test3_records.json", "w") as f:
    json.dump(all_patches, f)
print("Saved to", DRIVE_RESULTS)

In [ ]:
import os, sys, json
from collections import defaultdict
import numpy as np
import torch
from PIL import Image
from scipy.ndimage import sobel, laplace
import matplotlib.pyplot as plt

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

DCE_net = model.enhance_net_nopool().cuda().eval()
DCE_net.load_state_dict(torch.load("/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth", map_location="cuda"))

PATCH, GRAD_PERCENTILE = 16, 30
DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/diagnostics"

with open(f"{DRIVE_RESULTS}/patch_records.json") as f:
    patch_records = json.load(f)

sums = defaultdict(list)
for p in patch_records:
    amp = p["noise"][8] / (p["noise"][0] + 1e-6)
    sums[p["file"]].append(np.log(amp + 1e-3))
file_scores = {f: np.mean(v) for f, v in sums.items()}
worst_files = sorted(file_scores, key=file_scores.get, reverse=True)[:3]
print("Worst 3 files by mean noise-amplification:", worst_files)

def find_path(fname):
    for d in ["data/test_data/DICM", "data/test_data/LIME"]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(fname)

def to_gray(a): return a.mean(axis=2)
def crop_to_grid(a, patch=PATCH):
    H, W = a.shape[:2]
    return a[: (H // patch) * patch, : (W // patch) * patch]
def patch_reduce(a, patch, fn):
    H, W = a.shape
    nph, npw = H // patch, W // patch
    r = a[: nph * patch, : npw * patch].reshape(nph, patch, npw, patch)
    return fn(r, axis=(1, 3))

def run_model(image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, _, r = DCE_net(t)
    r_iters = r.squeeze(0).cpu().numpy().reshape(8, 3, *r.shape[-2:])
    x = t.squeeze(0).cpu().numpy()
    stages = [x]
    for i in range(8):
        x = x + r_iters[i] * (np.power(x, 2) - x)
        stages.append(x.copy())
    return [np.transpose(s, (1, 2, 0)) for s in stages]

MARGIN_PATCHES = 3
fig, axes = plt.subplots(len(worst_files), 2, figsize=(8, 4 * len(worst_files)))

for row, fname in enumerate(worst_files):
    stages_hwc = run_model(find_path(fname))
    img_in, img_out = crop_to_grid(stages_hwc[0]), crop_to_grid(stages_hwc[-1])
    gray_in = to_gray(img_in)
    grad = np.hypot(sobel(gray_in, axis=1), sobel(gray_in, axis=0))
    flat_mask = patch_reduce(grad, PATCH, np.mean) <= np.percentile(patch_reduce(grad, PATCH, np.mean), GRAD_PERCENTILE)
    noise_in_patch = patch_reduce(laplace(gray_in), PATCH, np.std)
    noise_out_patch = patch_reduce(laplace(to_gray(img_out)), PATCH, np.std)
    amp_map = noise_out_patch / (noise_in_patch + 1e-6)
    amp_masked = np.where(flat_mask, amp_map, -np.inf)
    i, j = np.unravel_index(np.argmax(amp_masked), amp_masked.shape)

    y0, y1 = max(0, (i - MARGIN_PATCHES) * PATCH), min(img_in.shape[0], (i + MARGIN_PATCHES + 1) * PATCH)
    x0p, x1 = max(0, (j - MARGIN_PATCHES) * PATCH), min(img_in.shape[1], (j + MARGIN_PATCHES + 1) * PATCH)
    crop_in = np.clip(img_in[y0:y1, x0p:x1], 0, 1)
    crop_out = np.clip(img_out[y0:y1, x0p:x1], 0, 1)

    axes[row, 0].imshow(crop_in); axes[row, 0].set_title(f"{fname} input (worst patch)"); axes[row, 0].axis("off")
    axes[row, 1].imshow(crop_out); axes[row, 1].set_title(f"{fname} enhanced ({amp_map[i,j]:.1f}x noise)"); axes[row, 1].axis("off")
    Image.fromarray((crop_in * 255).astype(np.uint8)).save(f"{DRIVE_RESULTS}/{fname}_worst_crop_input.png")
    Image.fromarray((crop_out * 255).astype(np.uint8)).save(f"{DRIVE_RESULTS}/{fname}_worst_crop_enhanced.png")

plt.tight_layout()
plt.savefig(f"{DRIVE_RESULTS}/test5_worst_crops.png", dpi=150)
plt.show()
print("Saved crops to", DRIVE_RESULTS)

In [ ]:
!pip install -q pyiqa

import os, sys, glob, json, statistics
import torch
import numpy as np
from PIL import Image
import pyiqa

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

DCE_net = model.enhance_net_nopool().cuda().eval()
DCE_net.load_state_dict(torch.load("/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth", map_location="cuda"))

available = pyiqa.list_models()
assert "niqe" in available and "brisque" in available, "niqe/brisque not in pyiqa.list_models()"
niqe_metric = pyiqa.create_metric("niqe", device="cuda")
brisque_metric = pyiqa.create_metric("brisque", device="cuda")

def enhance(image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, _ = DCE_net(t)
    return t, enhanced.clamp(0, 1)

test_images = sorted(glob.glob("data/test_data/DICM/*.jpg")) + sorted(glob.glob("data/test_data/LIME/*.bmp"))
print(f"{len(test_images)} test images")

records = []
for path in test_images:
    t_in, t_out = enhance(path)
    with torch.no_grad():
        niqe_in, niqe_out = niqe_metric(t_in).item(), niqe_metric(t_out).item()
        brisque_in, brisque_out = brisque_metric(t_in).item(), brisque_metric(t_out).item()
    records.append({"file": os.path.basename(path),
                     "niqe_in": niqe_in, "niqe_out": niqe_out, "niqe_delta": niqe_out - niqe_in,
                     "brisque_in": brisque_in, "brisque_out": brisque_out, "brisque_delta": brisque_out - brisque_in})

mean_niqe_in = statistics.mean(r["niqe_in"] for r in records)
mean_niqe_out = statistics.mean(r["niqe_out"] for r in records)
mean_brisque_in = statistics.mean(r["brisque_in"] for r in records)
mean_brisque_out = statistics.mean(r["brisque_out"] for r in records)
worse_niqe = [r for r in records if r["niqe_delta"] > 0]
worse_brisque = [r for r in records if r["brisque_delta"] > 0]

print(f"NIQE:    input mean {mean_niqe_in:.3f}  ->  enhanced mean {mean_niqe_out:.3f}  (lower=better)")
print(f"BRISQUE: input mean {mean_brisque_in:.3f}  ->  enhanced mean {mean_brisque_out:.3f}  (lower=better)")
print(f"{len(worse_niqe)}/{len(records)} images got WORSE NIQE after enhancement")
print(f"{len(worse_brisque)}/{len(records)} images got WORSE BRISQUE after enhancement")

print("\nWorst NIQE degradations:")
for r in sorted(records, key=lambda r: r["niqe_delta"], reverse=True)[:5]:
    print(f"  {r['file']}: {r['niqe_in']:.2f} -> {r['niqe_out']:.2f}  (delta={r['niqe_delta']:+.2f})")

worst_noise_files = {"18.jpg", "26.jpg", "14.jpg"}  # from Test 5
print("\nCross-check vs Test 5's worst noise-amplification files:")
for r in records:
    if r["file"] in worst_noise_files:
        print(f"  {r['file']}: NIQE {r['niqe_in']:.2f}->{r['niqe_out']:.2f} (d{r['niqe_delta']:+.2f})  BRISQUE {r['brisque_in']:.2f}->{r['brisque_out']:.2f} (d{r['brisque_delta']:+.2f})")

DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/diagnostics"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
with open(f"{DRIVE_RESULTS}/test4_iqa_records.json", "w") as f:
    json.dump(records, f, indent=2)
print("Saved to", DRIVE_RESULTS)

## Part 2 — `W_noise` sweep v1, weights 0.1–2.0 (cells 22–24)

Too small to have any effect.

In [ ]:
import os, sys, argparse, importlib, time
sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")

import Myloss
importlib.reload(Myloss)
import lowlight_train
importlib.reload(lowlight_train)

assert hasattr(Myloss, "L_noise"), "Myloss still missing L_noise after reload — something else is stale"
print("Myloss.L_noise is present, good to go")

BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
assert os.path.exists(BASELINE_CKPT), "Tier-1 baseline checkpoint not found on Drive"

SWEEP_DIR = "/content/drive/MyDrive/zero-dce/noise_sweep"
os.makedirs(SWEEP_DIR, exist_ok=True)

W_NOISE_VALUES = [0.1, 0.3, 0.6, 1.0, 2.0]
SWEEP_EPOCHS = 15

for W in W_NOISE_VALUES:
    tag = f"W{W}"
    out_dir = f"{SWEEP_DIR}/{tag}"
    os.makedirs(out_dir, exist_ok=True)
    final_ckpt = f"{out_dir}/Epoch{SWEEP_EPOCHS - 1}.pth"

    if os.path.exists(final_ckpt):
        print(f"[{tag}] already done, skipping")
        continue

    print(f"[{tag}] fine-tuning {SWEEP_EPOCHS} epochs from baseline, W_noise={W}")
    config = argparse.Namespace(
        lowlight_images_path="data/train_data/",
        lr=0.0001, weight_decay=0.0001, grad_clip_norm=0.1,
        num_epochs=SWEEP_EPOCHS, start_epoch=0,
        train_batch_size=8, val_batch_size=4, num_workers=4,
        display_iter=200, snapshot_iter=200,
        snapshots_folder=out_dir + "/",
        load_pretrain=True, pretrain_dir=BASELINE_CKPT,
        W_noise=W,
    )
    t0 = time.time()
    lowlight_train.train(config)
    print(f"[{tag}] done in {(time.time() - t0) / 60:.1f} min")

print("Sweep training complete.")

In [ ]:
!pip install -q pyiqa scikit-image

import os, sys, json, glob
import numpy as np
import torch
from PIL import Image
from scipy.ndimage import sobel, laplace
from scipy.stats import pearsonr
from skimage.metrics import peak_signal_noise_ratio as psnr_fn, structural_similarity as ssim_fn
import pyiqa

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

SWEEP_DIR = "/content/drive/MyDrive/zero-dce/noise_sweep"
W_NOISE_VALUES = [0.1, 0.3, 0.6, 1.0, 2.0]
SWEEP_EPOCHS = 15
BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
PATCH, GRAD_PERCENTILE = 16, 30

niqe_metric = pyiqa.create_metric("niqe", device="cuda")
brisque_metric = pyiqa.create_metric("brisque", device="cuda")

test_images = sorted(glob.glob("data/test_data/DICM/*.jpg")) + sorted(glob.glob("data/test_data/LIME/*.bmp"))

LOL_ROOT = "/content/lol_dataset/lol_dataset/eval15"
if not os.path.isdir(LOL_ROOT):
    from huggingface_hub import hf_hub_download
    import shutil
    zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset", filename="lol_dataset.zip", repo_type="dataset")
    shutil.unpack_archive(zip_path, "/content/lol_dataset")
LOW_DIR, HIGH_DIR = f"{LOL_ROOT}/low", f"{LOL_ROOT}/high"
lol_files = sorted(f for f in os.listdir(LOW_DIR) if not f.startswith("."))

def to_gray(a): return a.mean(axis=2)
def crop_to_grid(a, patch=PATCH):
    H, W = a.shape[:2]
    return a[: (H // patch) * patch, : (W // patch) * patch]
def patch_reduce(a, patch, fn):
    H, W = a.shape
    nph, npw = H // patch, W // patch
    r = a[: nph * patch, : npw * patch].reshape(nph, patch, npw, patch)
    return fn(r, axis=(1, 3))

def load_net(ckpt_path):
    net = model.enhance_net_nopool().cuda().eval()
    net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    return net

def run_model(net, image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, r = net(t)
    r_iters = r.squeeze(0).cpu().numpy().reshape(8, 3, *r.shape[-2:])
    x_prev = t.squeeze(0).cpu().numpy()
    for i in range(8):
        x_prev = x_prev + r_iters[i] * (np.power(x_prev, 2) - x_prev)
    out = np.transpose(x_prev, (1, 2, 0))
    avg_alpha = np.abs(r_iters).mean(axis=(0, 1))
    return t, enhanced.clamp(0, 1), x0, out, avg_alpha

def evaluate_candidate(ckpt_path, label):
    net = load_net(ckpt_path)
    amp_records = []
    niqe_in_list, niqe_out_list, brisque_in_list, brisque_out_list = [], [], [], []
    for path in test_images:
        t_in, t_out, x0, out, avg_alpha = run_model(net, path)
        x0c, outc, avg_alpha_c = crop_to_grid(x0), crop_to_grid(out), crop_to_grid(avg_alpha)
        gray_in = to_gray(x0c)
        grad = np.hypot(sobel(gray_in, axis=1), sobel(gray_in, axis=0))
        grad_patch = patch_reduce(grad, PATCH, np.mean)
        flat_mask = grad_patch <= np.percentile(grad_patch, GRAD_PERCENTILE)
        noise_in_patch = patch_reduce(laplace(gray_in), PATCH, np.std)
        noise_out_patch = patch_reduce(laplace(to_gray(outc)), PATCH, np.std)
        alpha_patch = patch_reduce(avg_alpha_c, PATCH, np.mean)
        amp = noise_out_patch / (noise_in_patch + 1e-6)
        nph, npw = flat_mask.shape
        for i in range(nph):
            for j in range(npw):
                if flat_mask[i, j]:
                    amp_records.append((float(amp[i, j]), float(alpha_patch[i, j])))
        with torch.no_grad():
            niqe_in_list.append(niqe_metric(t_in).item())
            niqe_out_list.append(niqe_metric(t_out).item())
            brisque_in_list.append(brisque_metric(t_in).item())
            brisque_out_list.append(brisque_metric(t_out).item())

    amps = np.array([a for a, _ in amp_records])
    alphas = np.array([al for _, al in amp_records])
    log_amp = np.log(amps + 1e-3)
    r_alpha, _ = pearsonr(alphas, log_amp)

    psnrs, ssims, maes = [], [], []
    for fname in lol_files:
        low_img = Image.open(f"{LOW_DIR}/{fname}").convert("RGB")
        x0 = np.asarray(low_img) / 255.0
        t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
        with torch.no_grad():
            _, enhanced, _ = net(t)
        out = enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1)
        gt_img = Image.open(f"{HIGH_DIR}/{fname}").convert("RGB")
        if gt_img.size != (x0.shape[1], x0.shape[0]):
            gt_img = gt_img.resize((x0.shape[1], x0.shape[0]))
        gt = np.asarray(gt_img).astype(np.float64)
        out255 = (out * 255).astype(np.float64)
        maes.append(float(np.mean(np.abs(out255 - gt))))
        psnrs.append(float(psnr_fn(gt, out255, data_range=255)))
        ssims.append(float(ssim_fn(gt, out255, data_range=255, channel_axis=2)))

    return {
        "label": label,
        "median_amp_ratio": float(np.median(amps)),
        "pearson_r_amp_vs_alpha": float(r_alpha),
        "niqe_in": float(np.mean(niqe_in_list)), "niqe_out": float(np.mean(niqe_out_list)),
        "brisque_in": float(np.mean(brisque_in_list)), "brisque_out": float(np.mean(brisque_out_list)),
        "lol_psnr": float(np.mean(psnrs)), "lol_ssim": float(np.mean(ssims)), "lol_mae": float(np.mean(maes)),
    }

RESULTS_PATH = f"{SWEEP_DIR}/sweep_results.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}

if "baseline" not in results:
    print("Evaluating baseline (W_noise=0, Tier-1 model)...")
    results["baseline"] = evaluate_candidate(BASELINE_CKPT, "baseline (W=0)")
    json.dump(results, open(RESULTS_PATH, "w"), indent=2)

for W in W_NOISE_VALUES:
    tag = f"W{W}"
    if tag in results:
        continue
    ckpt = f"{SWEEP_DIR}/{tag}/Epoch{SWEEP_EPOCHS - 1}.pth"
    if not os.path.exists(ckpt):
        print(f"[{tag}] checkpoint not found, skipping")
        continue
    print(f"Evaluating {tag}...")
    results[tag] = evaluate_candidate(ckpt, tag)
    json.dump(results, open(RESULTS_PATH, "w"), indent=2)

print(f"\n{'label':14s} {'med_amp':>8s} {'r(amp,a)':>9s} {'NIQE in->out':>14s} {'BRISQUE in->out':>17s} {'LOL PSNR':>9s} {'LOL SSIM':>9s} {'LOL MAE':>8s}")
for key in ["baseline"] + [f"W{w}" for w in W_NOISE_VALUES]:
    if key not in results: continue
    r = results[key]
    print(f"{r['label']:14s} {r['median_amp_ratio']:8.2f} {r['pearson_r_amp_vs_alpha']:9.3f} "
          f"{r['niqe_in']:5.2f}->{r['niqe_out']:<5.2f} {r['brisque_in']:6.1f}->{r['brisque_out']:<6.1f} "
          f"{r['lol_psnr']:9.2f} {r['lol_ssim']:9.3f} {r['lol_mae']:8.2f}")

print("\nSaved to", RESULTS_PATH)

In [ ]:
import os, sys, importlib
sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")

import torch
import model, dataloader, Myloss
importlib.reload(model); importlib.reload(dataloader); importlib.reload(Myloss)

DCE_net = model.enhance_net_nopool().cuda()
DCE_net.load_state_dict(torch.load("/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth", map_location="cuda"))
DCE_net.eval()

train_dataset = dataloader.lowlight_loader("data/train_data/")
loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

L_color = Myloss.L_color()
L_spa = Myloss.L_spa()
L_exp = Myloss.L_exp(16, 0.6)
L_TV = Myloss.L_TV()
L_noise = Myloss.L_noise()

batch = next(iter(loader)).cuda()
with torch.no_grad():
    enhanced_image_1, enhanced_image, A = DCE_net(batch)
    raw_TV = L_TV(A).item()
    raw_spa = torch.mean(L_spa(enhanced_image, batch)).item()
    raw_col = torch.mean(L_color(enhanced_image)).item()
    raw_exp = torch.mean(L_exp(enhanced_image)).item()
    raw_noise = L_noise(batch, A).item()

print(f"{'term':10s} {'raw value':>12s} {'actual weight':>14s} {'weighted value':>15s}")
print(f"{'TV':10s} {raw_TV:12.6f} {200:14.1f} {200*raw_TV:15.6f}")
print(f"{'spa':10s} {raw_spa:12.6f} {1:14.1f} {raw_spa:15.6f}")
print(f"{'col':10s} {raw_col:12.6f} {5:14.1f} {5*raw_col:15.6f}")
print(f"{'exp':10s} {raw_exp:12.6f} {10:14.1f} {10*raw_exp:15.6f}")
print(f"{'noise (raw, unweighted)':10s} {raw_noise:.6f}")

total_others = 200*raw_TV + raw_spa + 5*raw_col + 10*raw_exp
print(f"\nSum of the 4 existing (already-weighted) terms: {total_others:.6f}")
print(f"Raw (unweighted) L_noise value:                  {raw_noise:.6f}")
print(f"Ratio raw L_noise / sum of others:                {100*raw_noise/total_others:.4f}%\n")

for W in [0.1, 0.3, 0.6, 1.0, 2.0, 10, 50, 100, 200]:
    weighted = W * raw_noise
    pct = 100 * weighted / (total_others + weighted)
    print(f"  W_noise={W:6.1f}  ->  weighted loss_noise={weighted:.6f}   ({pct:.2f}% of total loss)")

## Part 3 — `W_noise` sweep v2, weights 20–400 (cells 25–28)

Buggy mask; heavy dimming.

In [ ]:
import os, sys, argparse, importlib, time
sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")

import Myloss, lowlight_train
importlib.reload(Myloss)
importlib.reload(lowlight_train)
assert hasattr(Myloss, "L_noise")

BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
assert os.path.exists(BASELINE_CKPT), "Tier-1 baseline checkpoint not found on Drive"

SWEEP_DIR = "/content/drive/MyDrive/zero-dce/noise_sweep_v2"
os.makedirs(SWEEP_DIR, exist_ok=True)

W_NOISE_VALUES = [20, 50, 100, 200, 400]
SWEEP_EPOCHS = 15

for W in W_NOISE_VALUES:
    tag = f"W{W}"
    out_dir = f"{SWEEP_DIR}/{tag}"
    os.makedirs(out_dir, exist_ok=True)
    final_ckpt = f"{out_dir}/Epoch{SWEEP_EPOCHS - 1}.pth"

    if os.path.exists(final_ckpt):
        print(f"[{tag}] already done, skipping")
        continue

    print(f"[{tag}] fine-tuning {SWEEP_EPOCHS} epochs from baseline, W_noise={W}")
    config = argparse.Namespace(
        lowlight_images_path="data/train_data/",
        lr=0.0001, weight_decay=0.0001, grad_clip_norm=0.1,
        num_epochs=SWEEP_EPOCHS, start_epoch=0,
        train_batch_size=8, val_batch_size=4, num_workers=4,
        display_iter=200, snapshot_iter=200,
        snapshots_folder=out_dir + "/",
        load_pretrain=True, pretrain_dir=BASELINE_CKPT,
        W_noise=W,
    )
    t0 = time.time()
    lowlight_train.train(config)
    print(f"[{tag}] done in {(time.time() - t0) / 60:.1f} min")

print("Sweep v2 training complete.")

In [ ]:
import os, json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import statistics

from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/zero-dce"

def show_image(path, title=None, figsize=(11, 6)):
    if not os.path.exists(path):
        print(f"  [missing] {path}")
        return
    img = Image.open(path)
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

def hr(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

# ---------------------------------------------------------------
hr("STEP 3 — SANITY CHECK (pretrained Epoch99.pth on bundled DICM/LIME images)")
show_image(f"{DRIVE}/results/sanity_check/sanity_check_grid.png", "Sanity check: input vs Zero-DCE output")

# ---------------------------------------------------------------
hr("TABLE 6 substitute — full-reference metrics on LOL eval15 (NOT the paper's SICE Part2 — different test set)")
lol_path = f"{DRIVE}/results/lol_eval15/metrics.json"
if os.path.exists(lol_path):
    avg = json.load(open(lol_path))["average"]
    print(f"{'Metric':8s} {'Our Zero-DCE, LOL eval15 (n=15)':>32s} {'Paper Zero-DCE, SICE Part2 (n=767)':>38s}")
    print(f"{'PSNR':8s} {avg['psnr']:>32.2f} {'16.57':>38s}")
    print(f"{'SSIM':8s} {avg['ssim']:>32.3f} {'0.59':>38s}")
    print(f"{'MAE':8s} {avg['mae']:>32.2f} {'98.78':>38s}")
    print("\nNumbers are NOT directly comparable (different test sets) — see CHANGES.md for why SICE Part2 was unreachable.")
else:
    print("  [missing]", lol_path)

# ---------------------------------------------------------------
hr("TABLE 7 — own params / FLOPs / runtime (previously measured, deterministic except runtime)")
print(f"{'Metric':22s} {'This reproduction (T4)':>24s} {'Paper (2080Ti)':>16s}")
print(f"{'Params':22s} {'79,416':>24s} {'79,416 / 79,532*':>16s}")
print(f"{'FLOPs (1200x900x3)':22s} {'85.54 G':>24s} {'84.99 G':>16s}")
print(f"{'Runtime (naive timing)':22s} {'114.04 ms':>24s} {'2.5 ms':>16s}")
print("* paper's own Table 4 and Table 7 disagree with each other on params by 116 — pre-existing paper-internal inconsistency.")
print("  Runtime gap (~45x) likely reflects measurement methodology (CUDA-event timing, warm cudnn) not raw hardware alone.")

# ---------------------------------------------------------------
hr("FIG 3/6 style — curve parameter maps")
show_image(f"{DRIVE}/results/curve_maps/curve_param_maps.png", "Curve parameter maps (avg over 8 iterations, R/G/B)")

# ---------------------------------------------------------------
hr("DIAGNOSTIC TEST 1 — noise amplification vs. curve magnitude (74 DICM+LIME images)")
patch_path = f"{DRIVE}/results/diagnostics/patch_records.json"
if os.path.exists(patch_path):
    patch_records = json.load(open(patch_path))
    alpha = np.array([p["alpha"] for p in patch_records])
    darkness = np.array([p["darkness"] for p in patch_records])
    noise_in = np.array([p["noise"][0] for p in patch_records])
    noise_out = np.array([p["noise"][8] for p in patch_records])
    amp = noise_out / (noise_in + 1e-6)
    log_amp = np.log(amp + 1e-3)
    r_alpha, _ = pearsonr(alpha, log_amp)
    r_dark, _ = pearsonr(darkness, log_amp)
    print(f"n={len(patch_records)} flat patches")
    print(f"Pearson r(amplification, curve |A|) = {r_alpha:.3f}")
    print(f"Pearson r(amplification, darkness)  = {r_dark:.3f}")
    print(f"Median amplification ratio          = {np.median(amp):.2f}x")
else:
    print("  [missing]", patch_path)
show_image(f"{DRIVE}/results/diagnostics/test1_noise_vs_alpha.png", "Test 1: amplification vs curve/darkness")

# ---------------------------------------------------------------
hr("DIAGNOSTIC TEST 2 — noise growth across the 8 curve iterations")
if os.path.exists(patch_path):
    noise_matrix = np.array([p["noise"] for p in patch_records])
    mean_per_stage = noise_matrix.mean(axis=0)
    print("Mean flat-patch noise level by stage (0=input .. 8=final output):")
    for i, v in enumerate(mean_per_stage):
        print(f"  stage {i}: {v:.5f}")
    print("Stage-over-stage growth:", np.round(np.diff(mean_per_stage), 5))
show_image(f"{DRIVE}/results/diagnostics/test2_noise_per_iteration.png", "Test 2: noise growth per iteration")

# ---------------------------------------------------------------
hr("DIAGNOSTIC TEST 3 — stratified full-reference error on LOL eval15")
t3_path = f"{DRIVE}/results/diagnostics/test3_records.json"
if os.path.exists(t3_path):
    t3 = json.load(open(t3_path))
    darkness3 = np.array([p["darkness"] for p in t3])
    err3 = np.array([p["err"] for p in t3])
    flat3 = np.array([p["flat"] for p in t3])
    noise_in3 = np.array([p["noise_in"] for p in t3])
    r_dark3, _ = pearsonr(darkness3, err3)
    r_noise3, _ = pearsonr(noise_in3[flat3], err3[flat3])
    print(f"n={len(t3)} patches across 15 LOL eval15 images")
    print(f"Pearson r(error, darkness)            = {r_dark3:.3f}")
    print(f"Pearson r(error, input noise, flat)   = {r_noise3:.3f}  (n_flat={flat3.sum()})")
    print("Honest reading: noise amplification does NOT meaningfully drive full-reference error.")
else:
    print("  [missing]", t3_path)
show_image(f"{DRIVE}/results/diagnostics/test3_error_vs_darkness_noise.png", "Test 3: error vs darkness/noise")

# ---------------------------------------------------------------
hr("DIAGNOSTIC TEST 4 — NIQE/BRISQUE before vs. after enhancement")
t4_path = f"{DRIVE}/results/diagnostics/test4_iqa_records.json"
if os.path.exists(t4_path):
    t4 = json.load(open(t4_path))
    print(f"NIQE:    {statistics.mean(r['niqe_in'] for r in t4):.3f} -> {statistics.mean(r['niqe_out'] for r in t4):.3f}  (improved on average)")
    print(f"BRISQUE: {statistics.mean(r['brisque_in'] for r in t4):.3f} -> {statistics.mean(r['brisque_out'] for r in t4):.3f}  (worsened on average)")
    worse_brisque = sum(1 for r in t4 if r["brisque_out"] > r["brisque_in"])
    print(f"{worse_brisque}/{len(t4)} images got worse BRISQUE after enhancement")
else:
    print("  [missing]", t4_path)

# ---------------------------------------------------------------
hr("DIAGNOSTIC TEST 5 — zoomed worst-case crops (14/18/26.jpg)")
show_image(f"{DRIVE}/results/diagnostics/test5_worst_crops.png", "Test 5: worst noise-amplification crops")

# ---------------------------------------------------------------
hr("NOISE-GAIN REGULARIZER — sweep v1 (W_noise 0.1-2.0, scale too small — see loss-scale diagnostic)")
sweep_path = f"{DRIVE}/noise_sweep/sweep_results.json"
if os.path.exists(sweep_path):
    sweep = json.load(open(sweep_path))
    print(f"{'label':14s} {'med_amp':>8s} {'r(amp,a)':>9s} {'NIQE in->out':>14s} {'BRISQUE in->out':>17s} {'LOL PSNR':>9s} {'LOL SSIM':>9s} {'LOL MAE':>8s}")
    for key in ["baseline", "W0.1", "W0.3", "W0.6", "W1.0", "W2.0"]:
        if key not in sweep:
            continue
        r = sweep[key]
        print(f"{r['label']:14s} {r['median_amp_ratio']:8.2f} {r['pearson_r_amp_vs_alpha']:9.3f} "
              f"{r['niqe_in']:5.2f}->{r['niqe_out']:<5.2f} {r['brisque_in']:6.1f}->{r['brisque_out']:<6.1f} "
              f"{r['lol_psnr']:9.2f} {r['lol_ssim']:9.3f} {r['lol_mae']:8.2f}")
    print("\nFinding: essentially no movement across weights. Loss-scale check showed W_noise=2.0 (the top of this range)")
    print("was only 0.67% of the total training loss — the whole sweep range was ~100x too small to matter.")
else:
    print("  [missing]", sweep_path)

hr("END OF SUMMARY — corrected sweep (W_noise 20-400) not yet run/evaluated")

In [ ]:
!pip install -q pyiqa scikit-image

import os, sys, json, glob
import numpy as np
import torch
from PIL import Image
from scipy.ndimage import sobel, laplace
from scipy.stats import pearsonr
from skimage.metrics import peak_signal_noise_ratio as psnr_fn, structural_similarity as ssim_fn
import pyiqa

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

SWEEP_DIR = "/content/drive/MyDrive/zero-dce/noise_sweep_v2"
W_NOISE_VALUES = [20, 50, 100, 200, 400]
SWEEP_EPOCHS = 15
BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
PATCH, GRAD_PERCENTILE = 16, 30

niqe_metric = pyiqa.create_metric("niqe", device="cuda")
brisque_metric = pyiqa.create_metric("brisque", device="cuda")

test_images = sorted(glob.glob("data/test_data/DICM/*.jpg")) + sorted(glob.glob("data/test_data/LIME/*.bmp"))

LOL_ROOT = "/content/lol_dataset/lol_dataset/eval15"
if not os.path.isdir(LOL_ROOT):
    from huggingface_hub import hf_hub_download
    import shutil
    zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset", filename="lol_dataset.zip", repo_type="dataset")
    shutil.unpack_archive(zip_path, "/content/lol_dataset")
LOW_DIR, HIGH_DIR = f"{LOL_ROOT}/low", f"{LOL_ROOT}/high"
lol_files = sorted(f for f in os.listdir(LOW_DIR) if not f.startswith("."))

def to_gray(a): return a.mean(axis=2)
def crop_to_grid(a, patch=PATCH):
    H, W = a.shape[:2]
    return a[: (H // patch) * patch, : (W // patch) * patch]
def patch_reduce(a, patch, fn):
    H, W = a.shape
    nph, npw = H // patch, W // patch
    r = a[: nph * patch, : npw * patch].reshape(nph, patch, npw, patch)
    return fn(r, axis=(1, 3))

def load_net(ckpt_path):
    net = model.enhance_net_nopool().cuda().eval()
    net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    return net

def run_model(net, image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, r = net(t)
    r_iters = r.squeeze(0).cpu().numpy().reshape(8, 3, *r.shape[-2:])
    x_prev = t.squeeze(0).cpu().numpy()
    for i in range(8):
        x_prev = x_prev + r_iters[i] * (np.power(x_prev, 2) - x_prev)
    out = np.transpose(x_prev, (1, 2, 0))
    avg_alpha = np.abs(r_iters).mean(axis=(0, 1))
    return t, enhanced.clamp(0, 1), x0, out, avg_alpha

def evaluate_candidate(ckpt_path, label):
    net = load_net(ckpt_path)
    amp_records = []
    niqe_in_list, niqe_out_list, brisque_in_list, brisque_out_list = [], [], [], []
    for path in test_images:
        t_in, t_out, x0, out, avg_alpha = run_model(net, path)
        x0c, outc, avg_alpha_c = crop_to_grid(x0), crop_to_grid(out), crop_to_grid(avg_alpha)
        gray_in = to_gray(x0c)
        grad = np.hypot(sobel(gray_in, axis=1), sobel(gray_in, axis=0))
        grad_patch = patch_reduce(grad, PATCH, np.mean)
        flat_mask = grad_patch <= np.percentile(grad_patch, GRAD_PERCENTILE)
        noise_in_patch = patch_reduce(laplace(gray_in), PATCH, np.std)
        noise_out_patch = patch_reduce(laplace(to_gray(outc)), PATCH, np.std)
        alpha_patch = patch_reduce(avg_alpha_c, PATCH, np.mean)
        amp = noise_out_patch / (noise_in_patch + 1e-6)
        nph, npw = flat_mask.shape
        for i in range(nph):
            for j in range(npw):
                if flat_mask[i, j]:
                    amp_records.append((float(amp[i, j]), float(alpha_patch[i, j])))
        with torch.no_grad():
            niqe_in_list.append(niqe_metric(t_in).item())
            niqe_out_list.append(niqe_metric(t_out).item())
            brisque_in_list.append(brisque_metric(t_in).item())
            brisque_out_list.append(brisque_metric(t_out).item())

    amps = np.array([a for a, _ in amp_records])
    alphas = np.array([al for _, al in amp_records])
    log_amp = np.log(amps + 1e-3)
    r_alpha, _ = pearsonr(alphas, log_amp)

    psnrs, ssims, maes = [], [], []
    for fname in lol_files:
        low_img = Image.open(f"{LOW_DIR}/{fname}").convert("RGB")
        x0 = np.asarray(low_img) / 255.0
        t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
        with torch.no_grad():
            _, enhanced, _ = net(t)
        out = enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1)
        gt_img = Image.open(f"{HIGH_DIR}/{fname}").convert("RGB")
        if gt_img.size != (x0.shape[1], x0.shape[0]):
            gt_img = gt_img.resize((x0.shape[1], x0.shape[0]))
        gt = np.asarray(gt_img).astype(np.float64)
        out255 = (out * 255).astype(np.float64)
        maes.append(float(np.mean(np.abs(out255 - gt))))
        psnrs.append(float(psnr_fn(gt, out255, data_range=255)))
        ssims.append(float(ssim_fn(gt, out255, data_range=255, channel_axis=2)))

    return {
        "label": label,
        "median_amp_ratio": float(np.median(amps)),
        "pearson_r_amp_vs_alpha": float(r_alpha),
        "niqe_in": float(np.mean(niqe_in_list)), "niqe_out": float(np.mean(niqe_out_list)),
        "brisque_in": float(np.mean(brisque_in_list)), "brisque_out": float(np.mean(brisque_out_list)),
        "lol_psnr": float(np.mean(psnrs)), "lol_ssim": float(np.mean(ssims)), "lol_mae": float(np.mean(maes)),
    }

RESULTS_PATH = f"{SWEEP_DIR}/sweep_results.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}

if "baseline" not in results:
    print("Evaluating baseline (W_noise=0, Tier-1 model)...")
    results["baseline"] = evaluate_candidate(BASELINE_CKPT, "baseline (W=0)")
    json.dump(results, open(RESULTS_PATH, "w"), indent=2)

for W in W_NOISE_VALUES:
    tag = f"W{W}"
    if tag in results:
        continue
    ckpt = f"{SWEEP_DIR}/{tag}/Epoch{SWEEP_EPOCHS - 1}.pth"
    if not os.path.exists(ckpt):
        print(f"[{tag}] checkpoint not found, skipping")
        continue
    print(f"Evaluating {tag}...")
    results[tag] = evaluate_candidate(ckpt, tag)
    json.dump(results, open(RESULTS_PATH, "w"), indent=2)

print(f"\n{'label':14s} {'med_amp':>8s} {'r(amp,a)':>9s} {'NIQE in->out':>14s} {'BRISQUE in->out':>17s} {'LOL PSNR':>9s} {'LOL SSIM':>9s} {'LOL MAE':>8s}")
for key in ["baseline"] + [f"W{w}" for w in W_NOISE_VALUES]:
    if key not in results: continue
    r = results[key]
    print(f"{r['label']:14s} {r['median_amp_ratio']:8.2f} {r['pearson_r_amp_vs_alpha']:9.3f} "
          f"{r['niqe_in']:5.2f}->{r['niqe_out']:<5.2f} {r['brisque_in']:6.1f}->{r['brisque_out']:<6.1f} "
          f"{r['lol_psnr']:9.2f} {r['lol_ssim']:9.3f} {r['lol_mae']:8.2f}")

print("\nSaved to", RESULTS_PATH)

In [ ]:
import os, sys
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

SWEEP_DIR = "/content/drive/MyDrive/zero-dce/noise_sweep_v2"
BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
W_NOISE_VALUES = [20, 50, 100, 200, 400]
SWEEP_EPOCHS = 15

def load_net(ckpt_path):
    net = model.enhance_net_nopool().cuda().eval()
    net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    return net

def enhance(net, image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, _ = net(t)
    return np.clip(enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy(), 0, 1)

candidates = [("baseline", BASELINE_CKPT)] + [(f"W{w}", f"{SWEEP_DIR}/W{w}/Epoch{SWEEP_EPOCHS-1}.pth") for w in W_NOISE_VALUES]
nets = {label: load_net(ckpt) for label, ckpt in candidates}

sample_images = [
    "data/test_data/DICM/18.jpg",   # worst noise-amplification case from Test 5
    "data/test_data/DICM/01.jpg",   # typical case from the original sanity check
    "data/test_data/LIME/1.bmp",
]

n_cols = len(candidates) + 1  # + input
fig, axes = plt.subplots(len(sample_images), n_cols, figsize=(3.2 * n_cols, 3.2 * len(sample_images)))

for row, path in enumerate(sample_images):
    inp = np.asarray(Image.open(path).convert("RGB")) / 255.0
    axes[row, 0].imshow(inp)
    axes[row, 0].set_title(f"input\n{os.path.basename(path)}" if row == 0 else os.path.basename(path), fontsize=9)
    axes[row, 0].axis("off")
    for col, (label, _) in enumerate(candidates, start=1):
        out = enhance(nets[label], path)
        axes[row, col].imshow(out)
        axes[row, col].set_title(label if row == 0 else "", fontsize=9)
        axes[row, col].axis("off")
        # mean brightness as a quick numeric anchor alongside the visual
        axes[row, col].text(0.02, 0.95, f"mean={out.mean():.2f}", transform=axes[row, col].transAxes,
                             fontsize=8, color="yellow", va="top", ha="left",
                             bbox=dict(facecolor="black", alpha=0.5, pad=1))

plt.tight_layout()
DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/diagnostics"
plt.savefig(f"{DRIVE_RESULTS}/sweep_v2_visual_check.png", dpi=110)
plt.show()
print("Saved to", DRIVE_RESULTS)

## Part 4 — `W_noise` v3, fixed mask at W50 (cells 30–33)

In [ ]:
import os, argparse, time

BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
OUT_DIR = "/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/W50"
os.makedirs(OUT_DIR, exist_ok=True)
SWEEP_EPOCHS = 15

config = argparse.Namespace(
    lowlight_images_path="data/train_data/",
    lr=0.0001, weight_decay=0.0001, grad_clip_norm=0.1,
    num_epochs=SWEEP_EPOCHS, start_epoch=0,
    train_batch_size=8, val_batch_size=4, num_workers=4,
    display_iter=200, snapshot_iter=200,
    snapshots_folder=OUT_DIR + "/",
    load_pretrain=True, pretrain_dir=BASELINE_CKPT,
    W_noise=50,
)
t0 = time.time()
lowlight_train.train(config)
print(f"Done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
!pip install -q pyiqa scikit-image

import os, sys, json, glob
import numpy as np
import torch
from PIL import Image
from scipy.ndimage import sobel, laplace
from scipy.stats import pearsonr
from skimage.metrics import peak_signal_noise_ratio as psnr_fn, structural_similarity as ssim_fn
import pyiqa

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
OLD_W50_CKPT = "/content/drive/MyDrive/zero-dce/noise_sweep_v2/W50/Epoch14.pth"
NEW_W50_CKPT = "/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/W50/Epoch14.pth"
PATCH, GRAD_PERCENTILE = 16, 30

niqe_metric = pyiqa.create_metric("niqe", device="cuda")
brisque_metric = pyiqa.create_metric("brisque", device="cuda")
test_images = sorted(glob.glob("data/test_data/DICM/*.jpg")) + sorted(glob.glob("data/test_data/LIME/*.bmp"))

LOL_ROOT = "/content/lol_dataset/lol_dataset/eval15"
if not os.path.isdir(LOL_ROOT):
    from huggingface_hub import hf_hub_download
    import shutil
    zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset", filename="lol_dataset.zip", repo_type="dataset")
    shutil.unpack_archive(zip_path, "/content/lol_dataset")
LOW_DIR, HIGH_DIR = f"{LOL_ROOT}/low", f"{LOL_ROOT}/high"
lol_files = sorted(f for f in os.listdir(LOW_DIR) if not f.startswith("."))

def to_gray(a): return a.mean(axis=2)
def crop_to_grid(a, patch=PATCH):
    H, W = a.shape[:2]
    return a[: (H // patch) * patch, : (W // patch) * patch]
def patch_reduce(a, patch, fn):
    H, W = a.shape
    nph, npw = H // patch, W // patch
    r = a[: nph * patch, : npw * patch].reshape(nph, patch, npw, patch)
    return fn(r, axis=(1, 3))

def load_net(ckpt_path):
    net = model.enhance_net_nopool().cuda().eval()
    net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    return net

def run_model(net, image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, r = net(t)
    r_iters = r.squeeze(0).cpu().numpy().reshape(8, 3, *r.shape[-2:])
    x_prev = t.squeeze(0).cpu().numpy()
    for i in range(8):
        x_prev = x_prev + r_iters[i] * (np.power(x_prev, 2) - x_prev)
    out = np.transpose(x_prev, (1, 2, 0))
    avg_alpha = np.abs(r_iters).mean(axis=(0, 1))
    return t, enhanced.clamp(0, 1), x0, out, avg_alpha

def evaluate_candidate(ckpt_path, label):
    net = load_net(ckpt_path)
    amp_records = []
    niqe_in_list, niqe_out_list, brisque_in_list, brisque_out_list = [], [], [], []
    for path in test_images:
        t_in, t_out, x0, out, avg_alpha = run_model(net, path)
        x0c, outc, avg_alpha_c = crop_to_grid(x0), crop_to_grid(out), crop_to_grid(avg_alpha)
        gray_in = to_gray(x0c)
        grad = np.hypot(sobel(gray_in, axis=1), sobel(gray_in, axis=0))
        grad_patch = patch_reduce(grad, PATCH, np.mean)
        flat_mask = grad_patch <= np.percentile(grad_patch, GRAD_PERCENTILE)
        noise_in_patch = patch_reduce(laplace(gray_in), PATCH, np.std)
        noise_out_patch = patch_reduce(laplace(to_gray(outc)), PATCH, np.std)
        alpha_patch = patch_reduce(avg_alpha_c, PATCH, np.mean)
        amp = noise_out_patch / (noise_in_patch + 1e-6)
        nph, npw = flat_mask.shape
        for i in range(nph):
            for j in range(npw):
                if flat_mask[i, j]:
                    amp_records.append((float(amp[i, j]), float(alpha_patch[i, j])))
        with torch.no_grad():
            niqe_in_list.append(niqe_metric(t_in).item())
            niqe_out_list.append(niqe_metric(t_out).item())
            brisque_in_list.append(brisque_metric(t_in).item())
            brisque_out_list.append(brisque_metric(t_out).item())

    amps = np.array([a for a, _ in amp_records])
    alphas = np.array([al for _, al in amp_records])
    log_amp = np.log(amps + 1e-3)
    r_alpha, _ = pearsonr(alphas, log_amp)

    psnrs, ssims, maes = [], [], []
    for fname in lol_files:
        low_img = Image.open(f"{LOW_DIR}/{fname}").convert("RGB")
        x0 = np.asarray(low_img) / 255.0
        t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
        with torch.no_grad():
            _, enhanced, _ = net(t)
        out = enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1)
        gt_img = Image.open(f"{HIGH_DIR}/{fname}").convert("RGB")
        if gt_img.size != (x0.shape[1], x0.shape[0]):
            gt_img = gt_img.resize((x0.shape[1], x0.shape[0]))
        gt = np.asarray(gt_img).astype(np.float64)
        out255 = (out * 255).astype(np.float64)
        maes.append(float(np.mean(np.abs(out255 - gt))))
        psnrs.append(float(psnr_fn(gt, out255, data_range=255)))
        ssims.append(float(ssim_fn(gt, out255, data_range=255, channel_axis=2)))

    return {
        "label": label,
        "median_amp_ratio": float(np.median(amps)),
        "pearson_r_amp_vs_alpha": float(r_alpha),
        "niqe_in": float(np.mean(niqe_in_list)), "niqe_out": float(np.mean(niqe_out_list)),
        "brisque_in": float(np.mean(brisque_in_list)), "brisque_out": float(np.mean(brisque_out_list)),
        "lol_psnr": float(np.mean(psnrs)), "lol_ssim": float(np.mean(ssims)), "lol_mae": float(np.mean(maes)),
    }

results = {}
results["baseline"] = evaluate_candidate(BASELINE_CKPT, "baseline (W=0)")
results["old_W50"] = evaluate_candidate(OLD_W50_CKPT, "old W50 (buggy mask)")
results["new_W50"] = evaluate_candidate(NEW_W50_CKPT, "new W50 (fixed mask)")

print(f"\n{'label':24s} {'med_amp':>8s} {'r(amp,a)':>9s} {'NIQE in->out':>14s} {'BRISQUE in->out':>17s} {'LOL PSNR':>9s} {'LOL SSIM':>9s} {'LOL MAE':>8s}")
for key in ["baseline", "old_W50", "new_W50"]:
    r = results[key]
    print(f"{r['label']:24s} {r['median_amp_ratio']:8.2f} {r['pearson_r_amp_vs_alpha']:9.3f} "
          f"{r['niqe_in']:5.2f}->{r['niqe_out']:<5.2f} {r['brisque_in']:6.1f}->{r['brisque_out']:<6.1f} "
          f"{r['lol_psnr']:9.2f} {r['lol_ssim']:9.3f} {r['lol_mae']:8.2f}")

json.dump(results, open("/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/comparison_results.json", "w"), indent=2)
print("\nSaved to /content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/comparison_results.json")

In [ ]:
import os, sys
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
NEW_W50_CKPT = "/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/W50/Epoch14.pth"

def load_net(ckpt_path):
    net = model.enhance_net_nopool().cuda().eval()
    net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    return net

def enhance(net, image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, _ = net(t)
    return np.clip(enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy(), 0, 1)

candidates = [("baseline", load_net(BASELINE_CKPT)), ("new W50 (fixed)", load_net(NEW_W50_CKPT))]

sample_images = [
    "data/test_data/DICM/18.jpg",
    "data/test_data/DICM/01.jpg",
    "data/test_data/LIME/1.bmp",
]

n_cols = len(candidates) + 1
fig, axes = plt.subplots(len(sample_images), n_cols, figsize=(4 * n_cols, 4 * len(sample_images)))

for row, path in enumerate(sample_images):
    inp = np.asarray(Image.open(path).convert("RGB")) / 255.0
    axes[row, 0].imshow(inp)
    axes[row, 0].set_title(f"input\n{os.path.basename(path)}" if row == 0 else os.path.basename(path), fontsize=10)
    axes[row, 0].axis("off")
    for col, (label, net) in enumerate(candidates, start=1):
        out = enhance(net, path)
        axes[row, col].imshow(out)
        axes[row, col].set_title(label if row == 0 else "", fontsize=10)
        axes[row, col].axis("off")
        axes[row, col].text(0.02, 0.95, f"mean={out.mean():.3f}", transform=axes[row, col].transAxes,
                             fontsize=9, color="yellow", va="top", ha="left",
                             bbox=dict(facecolor="black", alpha=0.5, pad=1))

plt.tight_layout()
DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed"
plt.savefig(f"{DRIVE_RESULTS}/w50_fixed_visual_check.png", dpi=110)
plt.show()
print("Saved to", DRIVE_RESULTS)

In [ ]:
import os, sys, json
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

# --- numbers, from the comparison run already saved to Drive ---
results = json.load(open("/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/comparison_results.json"))
print(f"{'label':24s} {'med_amp':>8s} {'r(amp,a)':>9s} {'NIQE in->out':>14s} {'BRISQUE in->out':>17s} {'LOL PSNR':>9s} {'LOL SSIM':>9s} {'LOL MAE':>8s}")
for key in ["baseline", "old_W50", "new_W50"]:
    r = results[key]
    print(f"{r['label']:24s} {r['median_amp_ratio']:8.2f} {r['pearson_r_amp_vs_alpha']:9.3f} "
          f"{r['niqe_in']:5.2f}->{r['niqe_out']:<5.2f} {r['brisque_in']:6.1f}->{r['brisque_out']:<6.1f} "
          f"{r['lol_psnr']:9.2f} {r['lol_ssim']:9.3f} {r['lol_mae']:8.2f}")

# --- image, baseline vs new W50 (fixed mask) ---
BASELINE_CKPT = "/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth"
NEW_W50_CKPT = "/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed/W50/Epoch14.pth"

def load_net(ckpt_path):
    net = model.enhance_net_nopool().cuda().eval()
    net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
    return net

def enhance(net, image_path):
    img = Image.open(image_path).convert("RGB")
    x0 = np.asarray(img) / 255.0
    t = torch.from_numpy(x0).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, _ = net(t)
    return np.clip(enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy(), 0, 1)

candidates = [("baseline", load_net(BASELINE_CKPT)), ("new W50 (fixed)", load_net(NEW_W50_CKPT))]
sample_images = ["data/test_data/DICM/18.jpg", "data/test_data/DICM/01.jpg", "data/test_data/LIME/1.bmp"]

n_cols = len(candidates) + 1
fig, axes = plt.subplots(len(sample_images), n_cols, figsize=(4 * n_cols, 4 * len(sample_images)))
for row, path in enumerate(sample_images):
    inp = np.asarray(Image.open(path).convert("RGB")) / 255.0
    axes[row, 0].imshow(inp)
    axes[row, 0].set_title(f"input\n{os.path.basename(path)}" if row == 0 else os.path.basename(path), fontsize=10)
    axes[row, 0].axis("off")
    for col, (label, net) in enumerate(candidates, start=1):
        out = enhance(net, path)
        axes[row, col].imshow(out)
        axes[row, col].set_title(label if row == 0 else "", fontsize=10)
        axes[row, col].axis("off")
        axes[row, col].text(0.02, 0.95, f"mean={out.mean():.3f}", transform=axes[row, col].transAxes,
                             fontsize=9, color="yellow", va="top", ha="left",
                             bbox=dict(facecolor="black", alpha=0.5, pad=1))
plt.tight_layout()
DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/noise_sweep_v3_fixed"
plt.savefig(f"{DRIVE_RESULTS}/w50_fixed_visual_check.png", dpi=110)
plt.show()
print("Saved to", DRIVE_RESULTS)